# Notebook 05: Advanced Visualizations and Interpretability Surfaces

This notebook produces professional visual diagnostics required for portfolio-quality reporting.


In [ ]:
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.model_selection import train_test_split

from src.data_loader import load_isolet_dataset
from src.feature_selector import FeatureSelector
from src.visualization import (
    plot_before_after_comparison,
    plot_benchmark_comparison,
    plot_dimensionality_reduction,
    plot_feature_selection_funnel,
    plot_learning_curve,
)

warnings.filterwarnings('ignore')
np.random.seed(42)
sns.set_theme(style='whitegrid')

FIG_DIR = Path('outputs/figures')
FIG_DIR.mkdir(parents=True, exist_ok=True)
METRIC_DIR = Path('outputs/metrics')


In [ ]:
X, y, _ = load_isolet_dataset()
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)
X_train_fs, X_val_fs, y_train_fs, y_val_fs = train_test_split(
    X_train, y_train, test_size=0.2, random_state=42, stratify=y_train
)

selector = FeatureSelector(random_state=42)
selected_features = selector.pipeline(
    X_train_fs,
    y_train_fs,
    X_val=X_val_fs,
    y_val=y_val_fs,
    var_threshold=0.0,
    corr_threshold=0.95,
    corr_method='spearman',
    rfe_feat=180,
    rfe_step=20,
    rfe_use_cv=False,
    rfe_min_features=40,
    l1_C=1.0,
    mi_k=160,
    shap_k=120,
    verbose=False,
)

X_train_sel = X_train[selected_features]
print(f'Features: {X_train.shape[1]} -> {X_train_sel.shape[1]}')


## Dimensionality reduction visualization

**PCA vs feature selection tradeoff:**
- PCA compresses dimensions into transformed latent components.
- Feature selection keeps original variables, preserving business interpretability.


In [ ]:
plot_dimensionality_reduction(
    X_original=X_train,
    y=y_train,
    X_selected=X_train_sel,
    title='PCA Projection: Original vs Selected Feature Space',
)


In [ ]:
stage_counts = {
    'Initial': int(X_train.shape[1]),
    'Variance': selector.results_['variance_threshold']['features_kept'],
    'Correlation': selector.results_['correlation_filter']['features_kept'],
    'RFECV': selector.results_['rfe']['features_kept'],
    'L1': selector.results_['l1_selection']['features_kept'],
    'MI': selector.results_['mutual_information']['features_kept'],
    'SHAP': selector.results_['shap_selection']['features_kept'],
}
plot_feature_selection_funnel(stage_counts)

retention_df = pd.DataFrame(
    {
        'stage': list(stage_counts.keys()),
        'features': list(stage_counts.values()),
    }
)
retention_df['retention_pct'] = retention_df['features'] / retention_df['features'].iloc[0] * 100
retention_df


In [ ]:
benchmark_path = METRIC_DIR / 'benchmark_summary.csv'
if benchmark_path.exists():
    benchmark_df = pd.read_csv(benchmark_path)

    comparison_df = (
        benchmark_df.groupby(['condition'])[['accuracy', 'f1', 'recall', 'precision']]
        .mean()
        .reset_index()
        .sort_values('condition')
    )
    plot_before_after_comparison(comparison_df)

    plot_benchmark_comparison(
        benchmark_df[[c for c in ['tool', 'condition', 'accuracy', 'f1'] if c in benchmark_df.columns]],
        metric='accuracy',
    )

    model_for_curve = benchmark_df[benchmark_df['tool'] == 'ManualModel'].copy()
    model_for_curve.head()
else:
    print('benchmark_summary.csv not found. Run Notebook 04 first for benchmark visuals.')


In [ ]:
from sklearn.ensemble import RandomForestClassifier

lc_model = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1)
plot_learning_curve(
    lc_model,
    X_train_sel,
    y_train,
    cv=3,
    title='Learning Curve on Selected Features',
)


## Interpretation

- Funnel and benchmark visuals make the feature-selection decision auditable.
- PCA plots are useful for geometry intuition, but selected original features remain superior for stakeholder interpretability.
